# Digit Recognizer Neural Network
- Dataset: [Kaggle Digit Recognizer](https://www.kaggle.com/c/digit-recognizer)


### **Libraries**

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

### **Loading dataset**

In [2]:
# ----------------- Load --------------------
data = pd.read_csv("C://Users//waels//Desktop//college//train.csv")

### **Preprocessing**

In [9]:
imputer = SimpleImputer(strategy='mean')  # or 'median', 'most_frequent', 'constant'
X_imputed = imputer.fit_transform(X)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
data

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41996,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41997,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### **Split**

In [10]:
#data = data.sample(frac=0.1)
X = data.iloc[:, 1:].values  # Pixel values (assuming first column is label)
y = data.iloc[:, 0].values.reshape(-1, 1)  # Labels

# Normalize pixel values to [0, 1]
X = X / 255.0

# One-hot encode labels
encoder = OneHotEncoder(sparse_output=False)
y_onehot = encoder.fit_transform(y)

# Split into train/validation sets
# Split first into train+val vs test (80-20)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42
)

# Then split temp into train-val (75-25 of 80% = 60-20 overall)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

### **Activations**

In [11]:
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))  # Clip to prevent overflow

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - np.tanh(x)**2

def identity(x):
    return x

def identity_derivative(x):
    return np.ones_like(x)

In [12]:
ACTIVATIONS = {
    'relu': (relu, relu_derivative),
    'sigmoid': (sigmoid, sigmoid_derivative),
    'tanh': (tanh, tanh_derivative),
    'softmax': (softmax, None),  # Derivative handled specially in backprop
    'linear': (identity, identity_derivative)
}

### **Loss**

In [13]:
def cross_entropy_loss(y_hat, y):
    m = y.shape[0]
    log_likelihood = -np.log(y_hat[range(m), y.argmax(axis=1)])
    return np.sum(log_likelihood) / m

### **Accuracy**

In [14]:
def accuracy(y_pred, y_true):
    return np.mean(np.argmax(y_pred, axis=1) == np.argmax(y_true, axis=1)) * 100

### **Initialize parameters**

In [15]:
def __init__(self, input_size, hidden_size, output_size):
        # Initialize parameters
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))
    

### **Forward pass**

In [16]:
def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = relu(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = softmax(self.z2)
        return self.a2

def compute_loss(self, X, y):
        return cross_entropy_loss(self.forward(X), y)

### **backward**

In [17]:
def backward(self, X, y):
        m = X.shape[0]
        # Output layer gradients
        dz2 = self.a2 - y
        self.dW2 = np.dot(self.a1.T, dz2) / m
        self.db2 = np.sum(dz2, axis=0, keepdims=True) / m
        
        # Hidden layer gradients
        da1 = np.dot(dz2, self.W2.T)
        dz1 = da1 * relu_derivative(self.z1)
        self.dW1 = np.dot(X.T, dz1) / m
        self.db1 = np.sum(dz1, axis=0, keepdims=True) / m

### **Optimizers**

In [18]:
# ----------------- SGD  -------------------------
def update_params(self, learning_rate):
        self.W1 -= learning_rate * self.dW1
        self.b1 -= learning_rate * self.db1
        self.W2 -= learning_rate * self.dW2
        self.b2 -= learning_rate * self.db2

In [24]:
class NeuralNetworkSgd:
    def __init__(self, layers, activations):
        self.layers = layers
        self.activations = activations
        self.weights = []
        self.biases = []

        for i in range(len(layers) - 1):
            # Initialize weights with Xavier/Glorot initialization
            weight = np.random.randn(layers[i], layers[i + 1]) * np.sqrt(2. / layers[i]) 
            bias = np.zeros((1, layers[i + 1]))
            self.weights.append(weight)
            self.biases.append(bias)
    def relu(self, x):
        return np.maximum(0, x)

    def relu_derivative(self, x):
        return (x > 0).astype(float)

    def softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)


    def forward(self, x):
        self.zs = []
        self.activations_cache = [x]

        for i in range(len(self.weights)):
            # Reshape bias to ensure correct broadcasting during addition
            z = self.activations_cache[-1] @ self.weights[i] + self.biases[i]  
            self.zs.append(z)

            if self.activations[i] == 'relu':
                a = self.relu(z)
            elif self.activations[i] == 'softmax':
                a = self.softmax(z)
            elif self.activations[i] == 'tanh':
                a = np.tanh(z)
            else:
                raise ValueError(f"Unsupported activation: {self.activations[i]}")  

            self.activations_cache.append(a)

        return self.activations_cache[-1]

    def compute_loss(self, X, y_true):
        y_pred = self.forward(X)
        m = y_true.shape[0]
        return -np.sum(y_true * np.log(y_pred + 1e-8)) / m

    def backward(self, y_true):
        grads_w = []
        grads_b = []
        m = y_true.shape[0]

        delta = self.activations_cache[-1] - y_true

        for i in reversed(range(len(self.weights))):
            a_prev = self.activations_cache[i]
            dw = a_prev.T @ delta / m
            db = np.sum(delta, axis=0, keepdims=True) / m

            grads_w.insert(0, dw)
            grads_b.insert(0, db)

            if i > 0:
                if self.activations[i-1] == 'relu':
                    delta = (delta @ self.weights[i].T) * self.relu_derivative(self.zs[i-1])
                elif self.activations[i-1] == 'tanh':
                    delta = (delta @ self.weights[i].T) * (1 - np.tanh(self.zs[i-1])**2)

        return grads_w, grads_b

    def update_params_sgd(self, X, y, learning_rate):
        self.forward(X)
        grads_w, grads_b = self.backward(y)
        for i in range(len(self.weights)):
            self.weights[i] -= learning_rate * grads_w[i]
            self.biases[i] -= learning_rate * grads_b[i]

    

### **Train**

In [ ]:
class digitClassifier(NeuralNetworkSgd):


    def train(self, X_train, y_train, X_val, y_val, epochs=1000, learning_rate=0.001, patience=10):
        best_val_loss = float('inf')
        best_params = None
        patience_counter = 0

        self.history = {
            'train_loss': [],
            'val_loss': [],
            'epochs': []
        }

        batch_size = 32

        for epoch in range(epochs):
            indices = np.arange(X_train.shape[0])
            np.random.shuffle(indices)
            X_train_shuffled = X_train[indices]
            y_train_shuffled = y_train[indices]

            for i in range(0, X_train.shape[0], batch_size):
                X_batch = X_train_shuffled[i:i + batch_size]
                y_batch = y_train_shuffled[i:i + batch_size]
                self.update_params_sgd(X_batch, y_batch, learning_rate=learning_rate)

            train_loss = self.compute_loss(X_train, y_train)
            val_loss = self.compute_loss(X_val, y_val)

            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['epochs'].append(epoch)

            if epoch % 100 == 0:
                print(f"Epoch {epoch}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_params = {
                    'weights': [w.copy() for w in self.weights],
                    'biases': [b.copy() for b in self.biases]
                }
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}. Best val loss: {best_val_loss:.4f}")
                break

        if best_params is not None:
            self.weights = best_params['weights']
            self.biases = best_params['biases']

        return best_val_loss
    
    

def plot_loss_history(self):
    
        plt.figure(figsize=(10, 6))
        plt.plot(self.history['epochs'], self.history['train_loss'], label='Training Loss')
        plt.plot(self.history['epochs'], self.history['val_loss'], label='Validation Loss')
        plt.title('Loss During Training')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        plt.show()

def final_evaluate(model, X_test, y_test):
    test_loss = model.compute_loss(X_test, y_test)
    print(f"**FINAL TEST LOSS: {test_loss:.4f}**")

# Example usage
if __name__ == "__main__":
    # Assuming 28x28 images (784 pixels)
    input_size = X_train.shape[1]
    output_size = 10  # Digits 0-9
    
    # Define architecture with multiple hidden layers
    layer_sizes = [input_size, 256, 128, output_size]  # Input, hidden1, hidden2, output
    activations = ['relu', 'relu', 'softmax']  # One activation per layer (except input) 
    model = digitClassifier(layer_sizes, activations)
    best_loss = model.train(X_train, y_train, X_val, y_val, epochs=1000, learning_rate=0.001)
    print(f"Best validation loss: {best_loss:.4f}")
    
    # Example prediction
    sample = X_val[0:1]
    probas = model.forward(sample)
    prediction = np.argmax(probas)
    print(f"Predicted: {prediction}, True: {np.argmax(y_val[0])}")

Epoch 0: Train Loss: 1.6618, Val Loss: 1.6641
Epoch 100: Train Loss: 0.1363, Val Loss: 0.1770
Epoch 200: Train Loss: 0.0727, Val Loss: 0.1349
Epoch 300: Train Loss: 0.0429, Val Loss: 0.1203
Epoch 400: Train Loss: 0.0264, Val Loss: 0.1150
Early stopping at epoch 457. Best val loss: 0.1142
Best validation loss: 0.1142
Predicted: 3, True: 3


In [26]:
def evaluate_model(model, X, y):
    # Convert one-hot encoded labels back to class indices
    y_true = np.argmax(y, axis=1)
    predictions = np.argmax(model.forward(X), axis=1)
    
    # Calculate metrics
    accuracy = np.mean(predictions == y_true)
    confusion_matrix = pd.crosstab(y_true, predictions, 
                                  rownames=['True'], 
                                  colnames=['Predicted'])
    
    print(f"Accuracy: {accuracy:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix)
    return accuracy, confusion_matrix

# Usage after training
print("\nTraining Set Evaluation:")
train_acc, train_cm = evaluate_model(model, X_train, y_train)

print("\nValidation Set Evaluation:")
val_acc, val_cm = evaluate_model(model, X_val, y_val)

print("\Test Set Evaluation:")
val_acc, val_cm = evaluate_model(model, X_test, y_test)


Training Set Evaluation:
Accuracy: 0.9976

Confusion Matrix:
Predicted     0     1     2     3     4     5     6     7     8     9
True                                                                 
0          2515     0     0     0     0     0     1     0     0     1
1             0  2815     1     1     0     0     2     1     1     1
2             1     0  2467     0     0     0     0     5     1     0
3             0     1     1  2572     1     0     0     0     1     3
4             0     2     1     0  2441     0     0     1     1     2
5             0     0     0     0     0  2310     2     0     0     0
6             0     0     0     0     2     1  2517     0     2     0
7             0     4     1     0     0     0     0  2609     0     2
8             0     2     0     1     0     0     2     0  2397     0
9             0     0     0     1     4     1     0     5     1  2496

Validation Set Evaluation:
Accuracy: 0.9679

Confusion Matrix:
Predicted    0    1    2    3    4